In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
# from tensorflow import keras

# peppers = pd.read_csv("../data/labels.csv")
# df = pd.DataFrame(peppers)


In [4]:
from pathlib import Path

validation_accuracy = np.array([])

tf.random.set_seed(123)

project_dir = Path.cwd().parent
img_dir = project_dir / "img"
csv_dir = project_dir / "data"

img_height = 180
img_width = 180
batch_size = 32
seed = 123

train_dataset = tf.keras.utils.image_dataset_from_directory(
    img_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(180, 180),
    batch_size=32
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    img_dir,
    validation_split=0.2,
    subset="validation",
    seed=seed,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

class_names = train_dataset.class_names

AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.cache().prefetch(buffer_size=AUTOTUNE)

model = tf.keras.Sequential([
    tf.keras.layers.Rescaling(1./255, input_shape=(img_height, img_width, 3)),
    tf.keras.layers.RandomFlip('horizontal'),
    # the augmentations below all had a negative impact when allowing tf to pick train/validate/test
    # tf.keras.layers.RandomRotation(0.02),
    # tf.keras.layers.RandomZoom(0.05),
    # tf.keras.layers.GaussianNoise(0.1),
    tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(3, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=15
)

val_loss, val_acc = model.evaluate(validation_dataset, verbose=2)

validation_accuracy = np.append(validation_accuracy, val_acc)

# print("Augmented with random flip: ")
# print(f"\n\nValidation accuracy: {validation_accuracy.mean():.2f}")

model.save('model.h5')


# print("\nValidation accuracy:", val_acc)

NotFoundError: Could not find directory /Users/josephklinck/Desktop/ml-pyenv/img